<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 2 · Mathematical and Statistical Preliminaries

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals

Chapter 2 refreshes the quantitative toolbox underpinning the rest of the book. This notebook adds hands-on demonstrations so you can:

- translate vectors/matrices into NumPy objects,
- compute sample statistics, covariance matrices, and correlations straight from <code>data/pyaiam_eod.csv</code>,
- visualize eigenstructure and covariance ellipses, and
- run miniature Monte Carlo experiments to see estimation error in action.

You can rerun every cell in Google Colab without external dependencies beyond the repository data file.

### Getting Help with Python Basics
Need a refresher while working through the math? Keep these sections open next to the notebook:

- **Chapter 3 – Python Infrastructure** for environment setup, notebooks, and plotting primers.
- **Appendix B – NumPy & pandas Reference** for array broadcasting, rolling stats, and plotting idioms.
- **Appendix A – Mathematical Foundations** if you want derivations of the formulas used here.

Dip into them whenever a pattern or formula looks new.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"


## 1. Load Prices and Build Return Panels
We reuse the multi-asset panel from Chapter 1 but focus on statistical structure. The same helper will serve later chapters too.

_Need reminders on pandas IO? Revisit Chapter 3 and Appendix B for <code>read_csv</code>, indexing, and resampling patterns._

In [ ]:
prices = pd.read_csv(DATA_PATH, parse_dates=["Date"]).set_index("Date").sort_index()
prices.head()

In [ ]:
def compute_return_panels(price_frame: pd.DataFrame) -> tuple[pd.DataFrame,
pd.DataFrame]:
    filled = price_frame.ffill()
    log_ret = np.log(filled / filled.shift(1)).dropna()
    simple_ret = filled.pct_change().dropna()
    return log_ret, simple_ret

log_rets, simple_rets = compute_return_panels(prices)
log_rets.head()

## 2. Sample Moments and Distribution Diagnostics

We inspect mean/variance estimates and compare empirical return histograms to Gaussian fits for quick sanity checks.

_For vectorized NumPy/pandas math, Appendix B offers concise reminders on broadcasting and aggregation._

In [ ]:
summary_stats = pd.DataFrame({
    "mean": log_rets.mean() * 252,
    "vol": log_rets.std() * np.sqrt(252),
    "skew": log_rets.skew(),
    "kurtosis": log_rets.kurtosis(),
})
summary_stats

In [ ]:
asset = "AAPL"
series = log_rets[asset]
fig, ax = plt.subplots(figsize=(12, 5))
series.plot(kind="hist", bins=60, density=True, alpha=0.6, label=f"{asset} log-ret")
mu, sigma = series.mean(), series.std()
x = np.linspace(series.min(), series.max(), 200)
ax.plot(x, 1 / (sigma * np.sqrt(2 * np.pi)) * np.exp(-0.5 * ((x - mu) / sigma) ** 2),
 label="Normal fit", linewidth=1.2)
ax.set_title(f"{asset} Daily Log-Return Distribution")
ax.legend()
plt.show()

## 3. Linear Algebra Warm-Up with NumPy

### 3.1 Vector Operations
We map Chapter 2 notation (\(\mathbf{x}, \mathbf{w}\)) to concrete arrays.

In [ ]:
w = np.array([0.3, 0.2, 0.5])
mu = np.array([0.08, 0.05, 0.06])
sigma = np.array(
    [
        [0.10, 0.02, 0.01],
        [0.02, 0.05, 0.015],
        [0.01, 0.015, 0.04],
    ]
)
port_return = w @ mu
port_var = w @ sigma @ w
port_sd = np.sqrt(port_var)
print(f"Expected return: {port_return:.3f}, volatility: {port_sd:.3f}")

### 3.2 Covariance Matrix from Data

We compute the sample covariance matrix for our asset universe and relate it to the theoretical example.

_Details on <code>cov()</code> and dataframe alignment live in Appendix B._

In [ ]:
sample_cov = log_rets.cov() * 252
sample_cov.loc[["AAPL", "NVDA", "SPY"], ["AAPL", "NVDA", "SPY"]]

### 3.3 Covariance Ellipse Visualization

Two-asset covariance can be visualized as an ellipse whose axes follow eigenvectors of the covariance matrix.

In [ ]:
pair = ["AAPL", "TLT"]
daily_cov = log_rets[pair].cov().values
vals, vecs = np.linalg.eigh(daily_cov)
angle = np.degrees(np.arctan2(*vecs[:, 1][::-1]))
width, height = 2 * np.sqrt(vals)
fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(
    log_rets[pair[0]],
    log_rets[pair[1]],
    s=3,
    alpha=0.3,
)
ellipse = plt.matplotlib.patches.Ellipse(
    (log_rets[pair[0]].mean(), log_rets[pair[1]].mean()),
    width,
    height,
    angle=angle,
    edgecolor="crimson",
    facecolor="none",
    linewidth=1.2,
)
ax.add_patch(ellipse)
ax.set_xlabel(f"{pair[0]} log-returns")
ax.set_ylabel(f"{pair[1]} log-returns")
ax.set_title("Covariance Ellipse (AAPL vs. TLT)")
plt.show()

## 4. Monte Carlo: Sampling Portfolios from the Covariance Matrix

We can reuse the sample covariance to simulate return scenarios and observe the efficient frontier shape emerging from random weights.

In [ ]:
rng = np.random.default_rng(0)
assets = ["AAPL", "NVDA", "SPY", "GLD", "TLT"]
mu_vec = summary_stats.loc[assets, "mean"].values
cov_mat = sample_cov.loc[assets, assets].values
n_portfolios = 5000
weights = rng.dirichlet(np.ones(len(assets)), size=n_portfolios)
port_means = weights @ mu_vec
port_vols = np.sqrt(np.einsum("bi,ij,bj->b", weights, cov_mat, weights))
fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(port_vols, port_means, c=port_means / port_vols, cmap="viridis", s=6)
ax.set_xlabel("Volatility")
ax.set_ylabel("Expected Return")
ax.set_title("Monte Carlo Portfolio Cloud")
plt.show()

## 5. Law of Large Numbers & Sampling Error Demo

Even simple averages converge slowly when volatility is high. The cell below repeatedly resamples <code>AAPL</code> returns to highlight estimation risk.

In [ ]:
sample_sizes = np.arange(20, 261, 20)
means = []
true_mean = log_rets["AAPL"].mean()
rng = np.random.default_rng(42)
for n in sample_sizes:
    boot = rng.choice(log_rets["AAPL"], size=n, replace=True)
    means.append(boot.mean())
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(sample_sizes, means, marker="o", linewidth=1.1)
ax.axhline(true_mean, color="red", linestyle="--", linewidth=1)
ax.set_xlabel("Sample Size")
ax.set_ylabel("Bootstrapped Mean")
ax.set_title("Sampling Variability of Daily Mean (AAPL)")
plt.show()

## 6. Exercises

### Exercise 1 – Rolling Covariance Explorer
Write a function <code>rolling_cov(asset_x, asset_y, window=63)</code> that returns a pandas Series of rolling covariance and plot it alongside the long-run estimate.

<details>
<summary>Hint</summary>
Use <code>log_rets[[asset_x, asset_y]].rolling(window).cov().unstack()</code> and select the appropriate column.
</details>

### Exercise 2 – Eigen Portfolio Sanity Check
Construct the eigenvectors of <code>sample_cov</code> for the subset <code>assets</code> and verify that the first eigen-portfolio has the highest variance when normalized to sum to one.

<details>
<summary>Hint</summary>
Use <code>np.linalg.eigh</code>, then scale eigenvectors so their weights sum to 1 before computing variance via <code>w @ cov_mat @ w</code>.
</details>

### Exercise 3 – Monte Carlo Confidence Bands
Extend the Monte Carlo experiment to compute the 5th/95th percentile of portfolio returns over a one-month horizon by sampling from a multivariate normal distribution.

<details>
<summary>Hint</summary>
Draw scenarios with <code>rng.multivariate_normal(mu_vec / 12, cov_mat / 12, size=5000)</code>, convert to cumulative returns, and aggregate by weights.
</details>


## 7. Takeaways for Chapter 2

- NumPy and pandas make the Chapter 2 formulas executable with a few lines of code.
- Visual diagnostics (histograms, ellipses, Monte Carlo clouds) highlight the assumptions behind classical portfolio theory.
- Sampling noise is real—keep an eye on estimation error before feeding numbers into optimizers.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">